# EPV exploratory data analysis

Tables only (no plotting library in this environment). The functions live in `analytics/epv/eda.py`; this notebook calls them on the synthetic fixture season by default, so **everything shown here is fixture output, not a result**. Set `FIXTURES = False` and `DATA_ROOT` to the owner's data root to run the same cells on real data.

Leakage rules: the feature frame comes from `analytics.epv.features.build_features`, whose only column source is `analytics.schema.feature_allowlist("states_1hz")` and which passes `analytics.schema.assert_time_forward`; the targets `points_to_end` and `possession_end_kind` are read here only as the y of the tables.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd()))

from IPython.display import Markdown, display
from analytics.epv import eda, features as F

FIXTURES = True
DATA_ROOT = None
SEED, N_GAMES = 0, 4

tables = eda.load_tables(fixtures=FIXTURES, data_root=DATA_ROOT, seed=SEED, n_games=N_GAMES)
states = tables['states_1hz']
train = states[~states['heldout']].reset_index(drop=True)
print('fixture output, not a result' if FIXTURES else DATA_ROOT)
print(len(states), 'states,', states['sportvu_game'].nunique(), 'games,', int(states['heldout'].sum()), 'held-out states set aside')

## Input validation

Schema problems, nulls, out-of-range counts and duplicates per table (`analytics.epv.validate`).

In [ ]:
from analytics.epv import validate as V
display(Markdown(V.format_report(tables['reports'])))

## points_to_end by elapsed bucket

Mean points and P(points > 0) per bucket of elapsed time, with the share of each possession end kind. Training games only.

In [ ]:
display(Markdown(eda.markdown(eda.points_by_elapsed_bucket(train))))

## Feature distributions

The geometric features (the priors are fold-dependent and are built by the trainer; here they come from all training games, so they are in-sample for this table only).

In [ ]:
priors = F.Priors.from_training(train, tables['shots'][~tables['shots']['heldout']] if tables.get('shots') is not None else None, tables.get('rosters'))
feats = F.build_features(train, priors)
display(Markdown(eda.markdown(eda.feature_distributions(feats), nd=3)))

## Correlation with the target

Pearson and Spearman correlation of every feature with `points_to_end`, missing values dropped pairwise.

In [ ]:
display(Markdown(eda.markdown(eda.target_correlation(feats, train['points_to_end']), nd=3)))

## Shuffled-target check

Gradient boosting out of fold on the real target and on possession points permuted across possessions: the shuffled lift must sit at or below chance.

In [ ]:
display(Markdown(eda.markdown(eda.shuffled_check(tables, folds=3, seed=SEED, hgb_iter=60))))